# GRU Improved Baseline

양방향 GRU + Focal Loss + 노이즈 증강 + BY 방향 포함으로 성능을 높인 개선 baseline.

목표: Float F1 ≥ 0.95 / INT8 F1 ≥ 0.90

고정 조건:
- Experiment ID: `B-GRU-improved`
- Model: `Bidirectional GRU`  `[128, 64]`
- Preprocessing: `PP-filtered`
- Data scope: `all` (BY 포함)
- Focal Loss: alpha=0.25, gamma=2.0
- Noise augmentation: std=0.01
- Output: `results/baselines_phase0/B-GRU-improved/`


## 1. Drive 마운트와 프로젝트 루트


In [ ]:
from pathlib import Path
import json
import os
import subprocess
import sys

try:
    from google.colab import drive  # type: ignore
    drive.mount('/content/drive')
except Exception:
    print('Not running in Colab or Drive mount skipped.')

PROJECT_ROOT_OVERRIDE = ""
PROJECT_ROOT_CANDIDATES = [
    Path('/content/drive/MyDrive/Graduate-Project/Falling-Model-Development'),
    Path('/content/drive/MyDrive/Falling-Model-Development'),
    Path('/content/drive/MyDrive/졸업 과제/Falling-Model-Development'),
    Path('/content/drive/MyDrive/졸업 과제/Falling-Detection-Development'),
    Path.cwd(),
]

if PROJECT_ROOT_OVERRIDE.strip():
    PROJECT_ROOT = Path(PROJECT_ROOT_OVERRIDE).expanduser()
else:
    PROJECT_ROOT = next((p for p in PROJECT_ROOT_CANDIDATES if (p / 'scripts' / 'train_baseline.py').exists()), None)
    if PROJECT_ROOT is None:
        raise FileNotFoundError('Project root not found. Set PROJECT_ROOT_OVERRIDE.')

os.chdir(PROJECT_ROOT)

# Experiment identity
EXPERIMENT_ID = 'B-GRU-raw'
MODEL_TYPE = 'gru'
PREPROCESSING = 'raw'

# Dataset paths are intentionally declared in this notebook.
TRAIN_CSV = PROJECT_ROOT / 'dataset' / 'train.csv'
VAL_CSV = PROJECT_ROOT / 'dataset' / 'val.csv'
TEST_CSV = PROJECT_ROOT / 'dataset' / 'test.csv'

# Input/window policy
INPUT_MODE = 'split_csv'
FEATURE_SET = 'kp12'
LABEL_COLUMN = 'label'
POSITIVE_LABELS = [1]
LABEL_MODE = 'segment_max'
DATA_SCOPE = 'no_by'
WINDOW_START_SEC = 5.0
WINDOW_END_SEC = 9.0
TARGET_STEPS = 60
TRAIN_POSITIVE_STRIDE = 1
TRAIN_NEGATIVE_STRIDE = 5
EVAL_STRIDE = 1

# Training policy
BATCH_SIZE = 64
EPOCHS = 100
LEARNING_RATE = 0.001
SEED = 42
DROPOUT_RATE = 0.2
EARLY_STOP_PATIENCE = 10
THRESHOLD_COUNT = 19
MIN_VAL_RECALL = 0.0

# Model structure
TCN_CHANNELS = [32, 32, 64, 96]
TCN_DILATIONS = [1, 2, 4, 8]
TCN_KERNEL_SIZE = 3
GRU_UNITS = [64, 32]

# Quantization/evaluation policy
REPRESENTATIVE_SAMPLES = 256
QUANT_EVAL_MAX_WINDOWS = 5000

OUTPUT_ROOT = PROJECT_ROOT / 'results' / 'baselines_phase0'
RESULT_DIR = OUTPUT_ROOT / EXPERIMENT_ID

print('PROJECT_ROOT  =', PROJECT_ROOT)
print('EXPERIMENT_ID =', EXPERIMENT_ID)
print('TRAIN_CSV     =', TRAIN_CSV)
print('VAL_CSV       =', VAL_CSV)
print('TEST_CSV      =', TEST_CSV)
print('RESULT_DIR    =', RESULT_DIR)


## 2. 런타임 준비


In [ ]:
subprocess.run(
    [sys.executable, '-m', 'pip', 'install', '-q', 'tensorflow', 'pandas', 'scikit-learn', 'matplotlib'],
    check=False,
)

import pandas as pd
from IPython.display import Image, Markdown, display
print('Runtime ready.')


## 3. 실험 설정


In [ ]:
# ── 실험 고유 설정 ─────────────────────────────────────────────
EXPERIMENT_ID = 'B-GRU-improved'
MODEL_TYPE = 'gru'
PREPROCESSING = 'filtered'   # 필터링된 피처 (AHSSC 등 포함)

# Dataset paths
TRAIN_CSV = PROJECT_ROOT / 'dataset' / 'train.csv'
VAL_CSV   = PROJECT_ROOT / 'dataset' / 'val.csv'
TEST_CSV  = PROJECT_ROOT / 'dataset' / 'test.csv'

# ── 데이터 정책 ─────────────────────────────────────────────────
INPUT_MODE  = 'split_csv'
FEATURE_SET = 'kp12'
LABEL_COLUMN   = 'label'
POSITIVE_LABELS = [1]
LABEL_MODE  = 'segment_max'
DATA_SCOPE  = 'all'          # BY 방향 포함 (raw baseline은 no_by)
WINDOW_START_SEC = 5.0
WINDOW_END_SEC   = 9.0
TARGET_STEPS = 60
TRAIN_POSITIVE_STRIDE = 1
TRAIN_NEGATIVE_STRIDE = 5
EVAL_STRIDE = 1

# ── 학습 정책 ────────────────────────────────────────────────────
BATCH_SIZE  = 64
EPOCHS      = 150
LEARNING_RATE = 0.001
SEED        = 42
DROPOUT_RATE = 0.2
EARLY_STOP_PATIENCE = 15

# ── 모델 구조 ────────────────────────────────────────────────────
GRU_UNITS     = [128, 64]    # raw baseline [64, 32]보다 2배
BIDIRECTIONAL = True         # 양방향 GRU
TCN_CHANNELS  = [32, 32, 64, 96]
TCN_DILATIONS = [1, 2, 4, 8]
TCN_KERNEL_SIZE = 3

# ── 손실 / 증강 ──────────────────────────────────────────────────
FOCAL_LOSS  = True           # 클래스 불균형 대응
FOCAL_ALPHA = 0.25
FOCAL_GAMMA = 2.0
NOISE_STD   = 0.01           # 훈련 데이터 가우시안 노이즈

# ── Threshold / 양자화 ───────────────────────────────────────────
THRESHOLD_COUNT = 19
MIN_VAL_RECALL  = 0.0
REPRESENTATIVE_SAMPLES = 256
QUANT_EVAL_MAX_WINDOWS = 5000

OUTPUT_ROOT = PROJECT_ROOT / 'results' / 'baselines_phase0'
RESULT_DIR  = OUTPUT_ROOT / EXPERIMENT_ID

import pandas as pd
from IPython.display import Markdown, Image, display

print('PROJECT_ROOT  =', PROJECT_ROOT)
print('EXPERIMENT_ID =', EXPERIMENT_ID)
print('RESULT_DIR    =', RESULT_DIR)

## 4. 학습 실행

`SMOKE = True`로 바꾸면 빠른 테스트 실행이 가능합니다.

`VERBOSE = True`이면 모델 구조(summary)와 epoch별 로그를 출력합니다. `False`(기본값)이면 데이터 로딩·요약 로그만 표시됩니다.


In [ ]:
import re
from IPython.display import clear_output

SMOKE   = False
EXPORT_TFLITE = True
VERBOSE = False

def csv_ints(values):
    return ','.join(str(value) for value in values)

def run_streaming(cmd, verbose=VERBOSE):
    cmd_str = ' '.join(str(part) for part in cmd)
    print(cmd_str)
    process = subprocess.Popen(
        [str(part) for part in cmd],
        stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
        text=True, bufsize=1,
    )
    assert process.stdout is not None
    epoch_re = re.compile(r'epoch (\d+)/(\d+)\s+(.*)')
    header_lines = [cmd_str]
    for line in process.stdout:
        line = line.rstrip()
        m = epoch_re.search(line)
        if m:
            cur, total, rest = int(m.group(1)), int(m.group(2)), m.group(3)
            filled = 32 * cur // total
            bar = '\u2588' * filled + '\u2591' * (32 - filled)
            progress = f'[{bar}] {cur:>4}/{total}  {rest}'
            if verbose:
                print(progress)
            else:
                clear_output(wait=True)
                for h in header_lines:
                    print(h)
                print(progress)
        else:
            print(line)
            if not verbose:
                header_lines.append(line)
    code = process.wait()
    if code != 0:
        raise RuntimeError(f'Command failed: {code}')

cmd = [
    sys.executable, 'scripts/train_baseline.py',
    '--project-root', PROJECT_ROOT,
    '--output-root', OUTPUT_ROOT,
    '--experiment-id', EXPERIMENT_ID,
    '--model-type', MODEL_TYPE,
    '--preprocessing', PREPROCESSING,
    '--input-mode', INPUT_MODE,
    '--train-csv', TRAIN_CSV,
    '--val-csv', VAL_CSV,
    '--test-csv', TEST_CSV,
    '--feature-set', FEATURE_SET,
    '--label-column', LABEL_COLUMN,
    '--positive-labels', csv_ints(POSITIVE_LABELS),
    '--label-mode', LABEL_MODE,
    '--data-scope', DATA_SCOPE,
    '--window-start-sec', WINDOW_START_SEC,
    '--window-end-sec', WINDOW_END_SEC,
    '--target-steps', TARGET_STEPS,
    '--train-positive-stride', TRAIN_POSITIVE_STRIDE,
    '--train-negative-stride', TRAIN_NEGATIVE_STRIDE,
    '--eval-stride', EVAL_STRIDE,
    '--batch-size', BATCH_SIZE,
    '--epochs', EPOCHS,
    '--learning-rate', LEARNING_RATE,
    '--seed', SEED,
    '--dropout-rate', DROPOUT_RATE,
    '--early-stop-patience', EARLY_STOP_PATIENCE,
    '--threshold-count', THRESHOLD_COUNT,
    '--min-val-recall', MIN_VAL_RECALL,
    '--tcn-channels', csv_ints(TCN_CHANNELS),
    '--tcn-dilations', csv_ints(TCN_DILATIONS),
    '--tcn-kernel-size', TCN_KERNEL_SIZE,
    '--gru-units', csv_ints(GRU_UNITS),
    '--representative-samples', REPRESENTATIVE_SAMPLES,
    '--quant-eval-max-windows', QUANT_EVAL_MAX_WINDOWS,
    '--noise-std', NOISE_STD,
    '--focal-alpha', FOCAL_ALPHA,
    '--focal-gamma', FOCAL_GAMMA,
]
if BIDIRECTIONAL:
    cmd.append('--bidirectional')
if FOCAL_LOSS:
    cmd.append('--focal-loss')
if SMOKE:
    cmd.append('--smoke')
if not EXPORT_TFLITE:
    cmd.append('--no-export-tflite')
if not VERBOSE:
    cmd.append('--quiet')
run_streaming(cmd)

## 5. 성능 테이블 (Window / Video 단위)


In [ ]:
metrics_data = json.loads((RESULT_DIR / 'metrics.json').read_text())
rows = []
for key, item in metrics_data['metrics'].items():
    rows.append({
        'split': key,
        'accuracy': item.get('accuracy'),
        'precision': item.get('precision'),
        'recall': item.get('recall'),
        'f1': item.get('f1'),
        'auc_roc': item.get('auc_roc'),
        'pr_auc': item.get('pr_auc'),
        'n': (item.get('positive_support', 0) or 0) + (item.get('negative_support', 0) or 0),
    })
df_metrics = pd.DataFrame(rows)
display(df_metrics.style.highlight_max(subset=['f1', 'auc_roc'], color='lightgreen'))
display(Markdown(f"Selected threshold: `{metrics_data['threshold_selection']['threshold']:.3f}`"))
print(f"Window counts  — train:{metrics_data['split_sizes']['train']:,}  val:{metrics_data['split_sizes']['val']:,}  test:{metrics_data['split_sizes']['test']:,}")
print(f"Video counts   — train:{metrics_data['video_counts']['train']:,}  val:{metrics_data['video_counts']['val']:,}  test:{metrics_data['video_counts']['test']:,}")

## 6. 시각화


In [ ]:
PLOT_FILES = [
    ('training_curve.png', '학습 곡선'),
    ('threshold_sweep.png', 'Threshold Sweep (Val)'),
    ('window_distribution.png', 'Window 분포 (Fall / Non-Fall / Direction)'),
    ('confusion_matrix.png', 'Float Test — Confusion Matrix (Window)'),
    ('class_metrics.png', 'Float Test — Fall vs Non-Fall 지표 (Window)'),
    ('direction_recall.png', 'Float Test — Direction별 Recall (Window)'),
    ('roc_curve.png', 'Float Test — ROC Curve'),
    ('pr_curve.png', 'Float Test — PR Curve'),
    ('val_confusion_matrix.png', 'Float Val — Confusion Matrix (Window)'),
    ('val_class_metrics.png', 'Float Val — Fall vs Non-Fall 지표 (Window)'),
    ('video_test_confusion_matrix.png', 'Video-Level Test — Confusion Matrix'),
    ('video_test_class_metrics.png', 'Video-Level Test — Fall vs Non-Fall 지표'),
    ('video_val_confusion_matrix.png', 'Video-Level Val — Confusion Matrix'),
    ('video_val_class_metrics.png', 'Video-Level Val — Fall vs Non-Fall 지표'),
    ('int8_confusion_matrix.png', 'INT8 — Confusion Matrix'),
    ('int8_class_metrics.png', 'INT8 — Fall vs Non-Fall 지표'),
    ('int8_roc_curve.png', 'INT8 — ROC Curve'),
    ('int8_pr_curve.png', 'INT8 — PR Curve'),
]

for filename, title in PLOT_FILES:
    path = RESULT_DIR / filename
    if path.exists():
        display(Markdown(f'### {title}'))
        display(Image(filename=str(path)))

## 7. 산출물 위치


In [ ]:
print(RESULT_DIR)
for path in sorted(RESULT_DIR.glob('*')):
    print(path.name)